In [ ]:
#!python get_coords.py --image ../IIT_HAZY/vid3/gt/00000001.jpg

# To get coordinates of bounding box. Run it in terminal, if window dont open here

In [ ]:
# !python generate_gt_labels.py \
#     --gt_dir ../IIT_HAZY/vid8/test/gt  \
#     --dehaze_ckpt model_weights/model_80.pkl \
#     --track_ckpt model_weights/siamrpn_r50.pth \
#     --out_json gt_perfect_labels_vid8.json \
#     --init_box 604 624 908 422

# Cotraining

In [ ]:
!python joint_train_dehaze_tracker.py \
    --train_manifests gt_perfect_labels_vid3.json \
    --hazy_dirs ../IIT_HAZY/vid3/hazy \
    --gt_dirs ../IIT_HAZY/vid3/gt \
    --dehaze_ckpt model_weights/model_80.pkl \
    --track_ckpt model_weights/siamrpn_r50.pth \
    --lr_dehaze 1e-4 --lr_tracker 1e-3 --epochs 6 \
    --pixel_loss_weight 0.03 \
    --out_prefix checkpoints/vid3_full/vid3_full_v1 \
    --pixel_loss_weight 0.03 --log_every 10 \
    --batch_size 1 --gpu 2
    # --max_frames 600 

In [ ]:
# Tracking after dehazing

!python eval_parallel.py \
    --dehaze_ckpt checkpoints/vid3_joint_shorted_IMP/vid3_shorted_v1_dehazer_epoch5.pkl \
    --track_ckpt checkpoints/vid3_joint_shorted_IMP/vid3_shorted_v1_tracker_epoch5.pth \
    --hazy_dir ../IIT_HAZY/vid8/test/hazy \
    --out_video vid8_test_11_shorted2.mp4 \
    --init_box 604 624 908 422 \
    --num_gpus 3 --start_gpu_id 1 \
    --gt_json gt_perfect_labels_vid8.json --save_json hazy_pred_labels_vid8_shorted.json \
    --max_frames 600
# confidence currently = 0.8

In [ ]:
!python results.py \
    --gt_json gt_perfect_labels_vid8.json \
    --pred_json hazy_baseline_results_vid8.json \
    --iou_threshold 0.5 \
    --center_threshold 20 \
    --csv_out new.csv \
    --plot_out new.png

In [ ]:
# Compare tree background
!python analyze_results.py \
    --gt_json gt_perfect_labels_vid3.json \
    --pred_json hazy_pred_labels_vid3.json \
    --conf_threshold 0.3 \
    --segment_start 250 --segment_end 370

!python analyze_results.py \
    --gt_json gt_perfect_labels_vid3.json \
    --pred_json hazy_baseline_results_vid3.json \
    --conf_threshold 0.3 \
    --segment_start 250 --segment_end 370    

In [ ]:
# Tracking after dehazing

!python eval_parallel.py \
    --dehaze_ckpt model_weights/model_80.pkl \
    --track_ckpt model_weights/siamrpn_r50.pth \
    --hazy_dir ../IIT_HAZY/vid3/hazy \
    --init_box 698 499 248 102 \
    --num_gpus 3 --start_gpu_id 1 \
    --gt_json gt_perfect_labels_vid3.json --save_json temp.json

In [ ]:
# Track hazy video directly with finetuned weights without dehazing
!python track_hazy.py \
    --track_ckpt checkpoints/siamrpn_test5.pth \
    --init_box 698 499 248 102 \
    --hazy_dir ../IIT_HAZY/vid3/hazy/ --save_json direct_finetuned_results_vid3.json \
    --out_video out_direct_track_test5_vid3.mp4 \
    --gpu 1

In [ ]:
# Track hazy video directly with finetuned weights without dehazing
!python track_hazy.py \
    --track_ckpt checkpoints/joint_v1_tracker_epoch2.pth \
    --init_box 698 499 248 102 \
    --hazy_dir ../IIT_HAZY/vid3/hazy/ --save_json temp.json \
    --out_video temp.mp4 \
    --gpu 1

!python results.py \
    --gt_json gt_perfect_labels_vid3.json \
    --pred_json temp.json \
    --iou_threshold 0.5 \
    --center_threshold 20 \
    --csv_out temp.csv \
    --plot_out temp.png    

In [ ]:
import csv
with open('per_frame_results_vid3.csv') as f:
    rows = [r for r in csv.DictReader(f) if float(r['iou']) <= 0.0]
    for r in rows:
        print(r['frame'], r['iou'])

#### Temp

In [ ]:
!python draw_box.py \
    --json_path hazy_baseline_results_vid3.json \
    --frames_dir ../IIT_HAZY/vid3/hazy \
    --output vid3_baseline.mp4 \
    --gpus 0,1,2,3


In [ ]:
!python eval_parallel.py \
    --dehaze_ckpt model_weights/model_80.pkl \
    --track_ckpt model_weights/siamrpn_r50.pth \
    --track_config pysot/experiments/siamrpn_r50_l234_dwxcorr/config.yaml \
    --hazy_dir ../IIT_HAZY/vid3/hazy \
    --init_box 698 499 248 102 \
    --max_frames 100 \
    --save_json results_pretrained.json \
    --gt_json gt_perfect_labels_vid3 \
    --out_video video_pretrained.mp4

In [ ]:
!python eval_parallel.py \
    --dehaze_ckpt checkpoints/joint_v1_dehazer_epoch2.pkl \
    --track_ckpt checkpoints/joint_v1_tracker_epoch2.pth \
    --track_config pysot/experiments/siamrpn_r50_l234_dwxcorr/config.yaml \
    --hazy_dir ../IIT_HAZY/vid3/hazy \
    --init_box 698 499 248 102 \
    --max_frames 100 \
    --save_json results_joint_epoch2.json \
    --gt_json results_pretrained.json \
    --out_video video_joint_epoch2.mp4